In [0]:

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

print("Spark Version:", spark.version)
print("Databricks environment ready")



Spark Version: 4.2.0
Databricks environment ready


In [0]:

from pyspark.sql.functions import col, current_timestamp

# Generate 5 simulated events per second
stream_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

# Display streaming source information
print("Streaming Source: Spark Rate")
print("Streaming Active:", stream_df.isStreaming)

stream_df.printSchema()


Streaming Source: Spark Rate
Streaming Active: True
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)



In [0]:

from pyspark.sql.functions import col, round as spark_round

# Transform incoming events into simulated transaction features
features_stream = (
    stream_df
    .withColumn(
        "transaction_amount",
        spark_round((col("value") % 200) * 2.5 + 50, 2)
    )
    .withColumn(
        "delivery_distance_km",
        spark_round((col("value") % 80) * 1.5 + 2, 2)
    )
    .select(
        "timestamp",
        "value",
        "transaction_amount",
        "delivery_distance_km"
    )
)

print("Streaming Transformation Successful")
print("Is Streaming:", features_stream.isStreaming)

features_stream.printSchema()


Streaming Transformation Successful
Is Streaming: True
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- delivery_distance_km: double (nullable = true)



In [0]:

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline
from pyspark.sql.functions import col

# Create representative historical simulated records
training_df = (
    spark.range(0, 1000)
    .withColumn(
        "transaction_amount",
        ((col("id") % 200) * 2.5 + 50).cast("double")
    )
    .withColumn(
        "delivery_distance_km",
        ((col("id") % 80) * 1.5 + 2).cast("double")
    )
)

# Combine numerical features
assembler = VectorAssembler(
    inputCols=["transaction_amount", "delivery_distance_km"],
    outputCol="raw_features"
)

# Standardise features
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

# Configure K-Means clustering
kmeans = KMeans(
    k=3,
    seed=42,
    featuresCol="features",
    predictionCol="cluster"
)

# Build and train the pipeline
pipeline = Pipeline(stages=[assembler, scaler, kmeans])
clustering_model = pipeline.fit(training_df)

print("K-Means Training Successful")
print("Training Records:", training_df.count())
print("Number of Clusters:", 3)


K-Means Training Successful
Training Records: 1000
Number of Clusters: 3


In [0]:

# Apply trained K-Means pipeline to streaming data
clustered_stream = clustering_model.transform(features_stream)

# Verify clustering output
print("Streaming Clustering Pipeline Created")
print("Is Streaming:", clustered_stream.isStreaming)

# Display output schema
clustered_stream.select(
    "timestamp",
    "transaction_amount",
    "delivery_distance_km",
    "cluster"
).printSchema()


Streaming Clustering Pipeline Created
Is Streaming: True
root
 |-- timestamp: timestamp (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- delivery_distance_km: double (nullable = true)
 |-- cluster: integer (nullable = false)



In [0]:

# Check the current catalog and schema
print("Current Catalog:")
spark.sql("SELECT current_catalog()").show()

print("Current Schema:")
spark.sql("SELECT current_schema()").show()

# Display available catalogs
print("Available Catalogs:")
spark.sql("SHOW CATALOGS").show(truncate=False)



Current Catalog:
+-----------------+
|current_catalog()|
+-----------------+
|        workspace|
+-----------------+

Current Schema:
+----------------+
|current_schema()|
+----------------+
|         default|
+----------------+

Available Catalogs:
+---------+
|catalog  |
+---------+
|samples  |
|system   |
|workspace|
+---------+



In [0]:

# Step 9: Verify integrated Olist dataset

olist_df = spark.table(
    "workspace.default.olist_integrated_clean"
)

print("Dataset loaded successfully!")
print("Total Records:", olist_df.count())
print("Total Columns:", len(olist_df.columns))

print("\nAvailable Columns:")
print(olist_df.columns)

# Check columns required for clustering
required_columns = [
    "delivery_days",
    "delivery_delay_days"
]

print("\nClustering Feature Verification:")
for column in required_columns:
    print(column, ":", column in olist_df.columns)

# Preview the dataset
display(olist_df.limit(10))


Dataset loaded successfully!
Total Records: 99441
Total Columns: 21

Available Columns:
['order_id', 'customer_id', 'order_status', 'purchase_date', 'approval_date', 'carrier_delivery_date', 'actual_delivery_date', 'estimated_delivery_date', 'approval_date_missing', 'carrier_date_missing', 'delivery_date_missing', 'purchase_year', 'purchase_month', 'purchase_weekday', 'delivery_days', 'delivery_delay_days', 'delivery_performance', 'customer_unique_id', 'zip_code', 'city', 'state']

Clustering Feature Verification:
delivery_days : True
delivery_delay_days : True


order_id,customer_id,order_status,purchase_date,approval_date,carrier_delivery_date,actual_delivery_date,estimated_delivery_date,approval_date_missing,carrier_date_missing,delivery_date_missing,purchase_year,purchase_month,purchase_weekday,delivery_days,delivery_delay_days,delivery_performance,customer_unique_id,zip_code,city,state
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18,0,0,0,2017,10,Monday,8.436574074074073,-7.107488425925926,On Time,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13,0,0,0,2018,7,Tuesday,13.782037037037037,-5.355729166666666,On Time,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04,0,0,0,2018,8,Wednesday,9.394212962962962,-17.245497685185185,On Time,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15,0,0,0,2017,11,Saturday,13.20875,-12.980069444444444,On Time,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26,0,0,0,2018,2,Tuesday,2.873877314814815,-9.238171296296295,On Time,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01,0,0,0,2017,7,Sunday,16.54224537037037,-5.543113425925926,On Time,80bb27c7c16e8f973207a5086ab329e2,86320,congonhinhas,PR
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09,0,1,1,2017,4,Tuesday,null,null,Not Delivered,36edbb3fb164b1f16485364b6fb04c73,98900,santa rosa,RS
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07,0,0,0,2017,5,Tuesday,9.989826388888888,-11.461215277777777,On Time,932afa1e708222e5821dac9cd5db4cae,26525,nilopolis,RJ
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06,0,0,0,2017,1,Monday,9.818761574074074,-31.41099537037037,On Time,39382392765b6dc74812866ee5ee92a7,99655,faxinalzinho,RS
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23,0,0,0,2017,7,Saturday,18.221851851851852,-6.2815972222222225,On Time,299905e3934e9e181bfb2e164dd4b4f8,18075,sorocaba,SP


In [0]:

from pyspark.sql.functions import col

# Load integrated Olist dataset
olist_df = spark.table(
    "workspace.default.olist_integrated_clean"
)

# Select valid delivered orders for clustering
olist_clustering_df = (
    olist_df
    .filter(col("order_status") == "delivered")
    .select(
        "order_id",
        col("delivery_days").cast("double").alias("delivery_days"),
        col("delivery_delay_days").cast("double").alias("delivery_delay_days")
    )
    .dropna(subset=["delivery_days", "delivery_delay_days"])
    .filter(col("delivery_days") >= 0)
)

# Verify prepared dataset
print("Olist Clustering Data Prepared Successfully")
print("Valid Records:", olist_clustering_df.count())

olist_clustering_df.printSchema()

display(olist_clustering_df.limit(10))


Olist Clustering Data Prepared Successfully
Valid Records: 96470
root
 |-- order_id: string (nullable = true)
 |-- delivery_days: double (nullable = true)
 |-- delivery_delay_days: double (nullable = true)



order_id,delivery_days,delivery_delay_days
e481f51cbdc54678b7cc49136f2d6af7,8.436574074074073,-7.107488425925926
53cdb2fc8bc7dce0b6741e2150273451,13.782037037037037,-5.355729166666666
47770eb9100c2d0c44946d9cf07ec65d,9.394212962962962,-17.245497685185185
949d5b44dbf5de918fe9c16f97b45f8a,13.20875,-12.980069444444444
ad21c59c0840e6cb83a9ceb5573f8159,2.873877314814815,-9.238171296296295
a4591c265e18cb1dcee52889e2d8acc3,16.54224537037037,-5.543113425925926
6514b8ad8028c9f2cc2374ded245783f,9.989826388888888,-11.461215277777777
76c6e866289321a7c93b82b54852dc33,9.818761574074074,-31.41099537037037
e69bfb5eb88e0ed6a785585b27e16dbf,18.221851851851852,-6.2815972222222225
e6ce16cb79ec1d90b1da9085a6118aeb,12.6509375,-8.52880787037037


In [0]:

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline

# Combine delivery features into a vector
assembler = VectorAssembler(
    inputCols=["delivery_days", "delivery_delay_days"],
    outputCol="raw_features"
)

# Standardise numerical features
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

# Configure K-Means with three clusters
kmeans = KMeans(
    k=3,
    seed=42,
    featuresCol="features",
    predictionCol="cluster"
)

# Build and train the clustering pipeline
olist_pipeline = Pipeline(
    stages=[assembler, scaler, kmeans]
)

olist_clustering_model = olist_pipeline.fit(
    olist_clustering_df
)

print("Olist K-Means Training Successful")
print("Training Records:", olist_clustering_df.count())
print("Number of Clusters:", 3)
print("Features: delivery_days, delivery_delay_days")


Olist K-Means Training Successful
Training Records: 96470
Number of Clusters: 3
Features: delivery_days, delivery_delay_days


In [0]:

from pyspark.sql.functions import count, avg, round as spark_round

# Apply the trained model to Olist records
olist_clustered_df = olist_clustering_model.transform(
    olist_clustering_df
)

# Calculate cluster distribution and average delivery metrics
cluster_summary = (
    olist_clustered_df
    .groupBy("cluster")
    .agg(
        count("*").alias("total_orders"),
        spark_round(avg("delivery_days"), 2).alias("avg_delivery_days"),
        spark_round(avg("delivery_delay_days"), 2).alias("avg_delay_days")
    )
    .orderBy("cluster")
)

print("Olist Cluster Distribution and Characteristics")
display(cluster_summary)


Olist Cluster Distribution and Characteristics


cluster,total_orders,avg_delivery_days,avg_delay_days
0,66419,8.34,-14.95
1,3594,43.8,16.81
2,26457,18.89,-5.52


In [0]:

from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# Assign sequential numbers to historical Olist orders
window_spec = Window.orderBy("order_id")

olist_stream_source = (
    olist_clustering_df
    .withColumn(
        "event_sequence",
        row_number().over(window_spec)
    )
)

# Save historical records as a Delta source table
olist_stream_source.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.olist_stream_source")

# Verify the source table
print("Olist Streaming Source Prepared Successfully")
print("Total Source Records:", spark.table(
    "workspace.default.olist_stream_source"
).count())

display(
    spark.table("workspace.default.olist_stream_source")
    .limit(10)
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Olist Streaming Source Prepared Successfully
Total Source Records: 96470


order_id,delivery_days,delivery_delay_days,event_sequence
00010242fe8c5a6d1ba2dd792cb16214,7.614421296296296,-8.01125,1
00018f77f2f0320c557190d7a144bdd3,16.216180555555557,-2.330277777777778,2
000229ec398224ef6ca0657da4fc703e,7.9484375,-13.444953703703703,3
00024acbcdf0a6daa1e931b038114c75,6.147268518518518,-5.435659722222222,4
00042b26cf59d7ce69dfabb4e55b4fd9,25.11435185185185,-15.30380787037037,5
00048cc3ae777c65dbb7d2a0634bc1ea,6.668067129629629,-14.427372685185185,6
00054e8431b9d7675808bcb819fb4a32,8.42349537037037,-16.080810185185186,7
000576fe39319847cbb9d288c5617fa6,5.080324074074074,-15.41380787037037,8
0005a1a1728c9d785b8e2b08b904576c,9.984004629629629,0.7621643518518518,9
0005f50442cb953dcd1d21e1fb923495,2.145046296296296,-18.271863425925925,10


In [0]:

# Step 14: Initialise Olist streaming clustering pipeline

# Read the Delta source as a streaming DataFrame
olist_live_stream = (
    spark.readStream
    .format("delta")
    .table("workspace.default.olist_stream_source")
)

# Apply the trained Olist K-Means model
olist_clustered_stream = olist_clustering_model.transform(
    olist_live_stream
)

# Verify streaming integration
print("Olist Streaming Clustering Pipeline Created")
print("Streaming Enabled:", olist_clustered_stream.isStreaming)

olist_clustered_stream.select(
    "order_id",
    "delivery_days",
    "delivery_delay_days",
    "event_sequence",
    "cluster"
).printSchema()


Olist Streaming Clustering Pipeline Created
Streaming Enabled: True
root
 |-- order_id: string (nullable = true)
 |-- delivery_days: double (nullable = true)
 |-- delivery_delay_days: double (nullable = true)
 |-- event_sequence: integer (nullable = true)
 |-- cluster: integer (nullable = false)



In [0]:

# Step 15: Execute Olist streaming clustering

output_table = "workspace.default.olist_stream_clusters"

checkpoint_path = (
    "/Volumes/workspace/default/"
    "bbim707_checkpoints/olist_clusters_v1"
)

# Select the required output fields
final_olist_stream = olist_clustered_stream.select(
    "order_id",
    "delivery_days",
    "delivery_delay_days",
    "event_sequence",
    "cluster"
)

# Process available records and save cluster assignments
olist_query = (
    final_olist_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(output_table)
)

# Wait for processing to complete
olist_query.awaitTermination()

print("Olist Streaming Execution Completed")
print("Query Active:", olist_query.isActive)
print("Output Table:", output_table)

# Verify saved clustering results
print("Processed Records:")
print(spark.table(output_table).count())

display(spark.table(output_table).limit(10))


Olist Streaming Execution Completed
Query Active: False
Output Table: workspace.default.olist_stream_clusters
Processed Records:
96470


order_id,delivery_days,delivery_delay_days,event_sequence,cluster
00010242fe8c5a6d1ba2dd792cb16214,7.614421296296296,-8.01125,1,0
00018f77f2f0320c557190d7a144bdd3,16.216180555555557,-2.330277777777778,2,2
000229ec398224ef6ca0657da4fc703e,7.9484375,-13.444953703703703,3,0
00024acbcdf0a6daa1e931b038114c75,6.147268518518518,-5.435659722222222,4,0
00042b26cf59d7ce69dfabb4e55b4fd9,25.11435185185185,-15.30380787037037,5,2
00048cc3ae777c65dbb7d2a0634bc1ea,6.668067129629629,-14.427372685185185,6,0
00054e8431b9d7675808bcb819fb4a32,8.42349537037037,-16.080810185185186,7,0
000576fe39319847cbb9d288c5617fa6,5.080324074074074,-15.41380787037037,8,0
0005a1a1728c9d785b8e2b08b904576c,9.984004629629629,0.7621643518518518,9,2
0005f50442cb953dcd1d21e1fb923495,2.145046296296296,-18.271863425925925,10,0


In [0]:

# Step 16: Verify processed streaming records

from pyspark.sql.functions import countDistinct

results_df = spark.table(
    "workspace.default.olist_stream_clusters"
)

total_records = results_df.count()
unique_orders = results_df.select(
    countDistinct("order_id")
).first()[0]

print("Streaming Output Verification")
print("Total Processed Records:", total_records)
print("Unique Order IDs:", unique_orders)
print("Duplicate Records:", total_records - unique_orders)

print("\nCluster Distribution:")
results_df.groupBy("cluster").count().orderBy("cluster").show()


Streaming Output Verification
Total Processed Records: 96470
Unique Order IDs: 96470
Duplicate Records: 0

Cluster Distribution:
+-------+-----+
|cluster|count|
+-------+-----+
|      0|66419|
|      1| 3594|
|      2|26457|
+-------+-----+



In [0]:

from pyspark.sql.functions import col, concat, lit

# Prepare 100 simulated new arrivals
new_olist_events = (
    spark.table("workspace.default.olist_stream_source")
    .orderBy("event_sequence")
    .limit(100)
    .withColumn(
        "order_id",
        concat(lit("SIM_"), col("order_id"))
    )
    .withColumn(
        "event_sequence",
        col("event_sequence") + 96470
    )
)

# Append simulated arrivals to the existing Delta source
new_olist_events.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.olist_stream_source")

print("Simulated New Events Added:", new_olist_events.count())

print(
    "Updated Source Record Count:",
    spark.table("workspace.default.olist_stream_source").count()
)


Simulated New Events Added: 100
Updated Source Record Count: 96570


In [0]:

from pyspark.sql.functions import countDistinct

# Load streaming output
results = spark.table(
    "workspace.default.olist_stream_clusters"
)

# Verify all processed records
total = results.count()
unique = results.select(
    countDistinct("order_id")
).first()[0]

# Verify simulated events
new_events = results.filter(
    "order_id LIKE 'SIM_%'"
).count()

print("Incremental Streaming Verification")
print("Total Records:", total)
print("Unique Orders:", unique)
print("Duplicate Records:", total - unique)
print("New Simulated Events:", new_events)

# Verify clustering of newly arrived events
print("\nNew Event Cluster Distribution:")
results.filter(
    "order_id LIKE 'SIM_%'"
).groupBy("cluster").count().orderBy("cluster").show()


Incremental Streaming Verification
Total Records: 96470
Unique Orders: 96470
Duplicate Records: 0
New Simulated Events: 0

New Event Cluster Distribution:
+-------+-----+
|cluster|count|
+-------+-----+
+-------+-----+

